In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#file_path_test1 = '/content/drive/MyDrive/Projects/Cybersecurity/GUIDE_Test1.csv'
#file_path_train = '/content/drive/MyDrive/Projects/Cybersecurity/new_train_sample.csv'
file_path_train2 = '/content/drive/MyDrive/Projects/Cybersecurity/new_train_sample1.csv'

Mounted at /content/drive


In [ ]:
selected_columns = [
    'IncidentGrade',  # 🎯 Target
    'Category',
    'AlertTitle',
    'MitreTechniques',
    'ActionGrouped',
    'ActionGranular',
    'EntityType',
    'EvidenceRole',
    'ThreatFamily',
    'OSFamily',
    'OSVersion',
    'SuspicionLevel',
    'CountryCode',
    'State',
    'City'
]

In [ ]:
# Step 1: Imports & data loading
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Read data (replace path with your actual files)
df = pd.read_csv(file_path_train2,usecols=selected_columns)

# 📊 Define Features and Target
X = df.drop('IncidentGrade', axis=1)
y = df['IncidentGrade']

# Encode target labels
le = LabelEncoder()
y = le.fit_transform(y)

# Split for validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, "Val:", X_val.shape)


Train: (3806734, 14) Val: (951684, 14)


In [ ]:
from scipy.stats.mstats import winsorize

X_train_capped = X_train.copy()
for col in X_train_capped.select_dtypes(include=['number']).columns:
    X_train_capped[col] = winsorize(X_train_capped[col], limits=[0.10, 0.10])  # Cap 1% tails


In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = X_train_capped.select_dtypes(include='number').columns
scaler = StandardScaler()
X_train_capped[num_cols] = scaler.fit_transform(X_train_capped[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])


In [ ]:
X_train_capped.select_dtypes(exclude=['number']).columns.tolist()


['Category',
 'MitreTechniques',
 'ActionGrouped',
 'ActionGranular',
 'EntityType',
 'EvidenceRole',
 'ThreatFamily',
 'SuspicionLevel']

In [ ]:
from sklearn.preprocessing import LabelEncoder

X_train_capped = X_train_capped.copy()
X_val = X_val.copy()

le_dict = {}

for col in X_train_capped.select_dtypes(exclude=['number']).columns:
    le = LabelEncoder()

    # Combine unique values from both train and val before fitting
    combined_vals = pd.concat([X_train_capped[col], X_val[col]], axis=0).astype(str)
    le.fit(combined_vals)

    # Transform both sets safely
    X_train_capped[col] = le.transform(X_train_capped[col].astype(str))
    X_val[col] = le.transform(X_val[col].astype(str))

    le_dict[col] = le

print("✅ Encoding complete for all categorical columns.")


✅ Encoding complete for all categorical columns.


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb


# List of models
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=10, min_samples_split=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=6, subsample=0.8, colsample_bytree=0.8,
        random_state=42, use_label_encoder=False, eval_metric='mlogloss'
    )
}

# Dictionary to store results
results = {}

for name, model in models.items():
    model.fit(X_train_capped, y_train)
    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='macro')
    precision = precision_score(y_val, y_pred, average='macro')
    recall = recall_score(y_val, y_pred, average='macro')

    results[name] = [accuracy, f1, precision, recall]

    print(f"{name} Results")
    print("-" * (len(name) + 8))
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("")

# Convert to DataFrame for easy comparison
import pandas as pd
df_results = pd.DataFrame(results, index=["Accuracy", "F1 Score", "Precision", "Recall"]).T
print("📊 Model Comparison:")
display(df_results.sort_values(by="F1 Score", ascending=False))

Decision Tree Results
---------------------
Accuracy: 0.6696
F1 Score: 0.6711
Precision: 0.7676
Recall: 0.6679

Random Forest Results
---------------------
Accuracy: 0.7043
F1 Score: 0.7227
Precision: 0.7896
Recall: 0.7231

Gradient Boosting Results
-------------------------
Accuracy: 0.7168
F1 Score: 0.7323
Precision: 0.7986
Recall: 0.7318



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression Results
---------------------------
Accuracy: 0.5382
F1 Score: 0.3408
Precision: 0.5567
Recall: 0.3602



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [13:47:15] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Results
---------------
Accuracy: 0.7120
F1 Score: 0.7320
Precision: 0.7849
Recall: 0.7329

📊 Model Comparison:


,Accuracy,F1 Score,Precision,Recall
Gradient Boosting,0.716780,0.732276,0.798574,0.731777
XGBoost,0.712025,0.732043,0.784906,0.732905
Random Forest,0.704324,0.722656,0.789598,0.723069
Decision Tree,0.669575,0.671124,0.767587,0.667884
Logistic Regression,0.538167,0.340830,0.556694,0.360153
